<a href="https://colab.research.google.com/github/Pablos29/prueba_tesis/blob/main/Calculo_PPSD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Calculo del Desplazamiento RMS del Movimiento del Suelo vs Tiempo**

*Este cuaderno de trabajo documenta el cálculo de los PPSD (Probability Power Spectral Densities) y la extracción del RMS sismico para el estudio del efecto del confinamiento durante la pandemia por COVID-19 en el centro historico de Querétaro, Mexico.*


Se requiere:
* Python
* ObsPy (y sus dependencias)
* Pandas
* Google Colab
* tqdm



Autor: Juan Pablo Sánchez (@Pablos29)

Este es un cuaderno de trabajo basado en el trabajo de:

Thomas Lecocq (@seismotom), Fred Massin (@fmassin), Claudio Satriano (@claudiodsf)

Para consultar el cuaderno original:
https://github.com/ThomasLecocq/SeismoRMS


# Configuración del entorno de trabajo


Este cuaderno de trabajo ha sido diseñado para su uso en Google Colab, por lo tanto, es necesario realizar algunas configuraciones para optimizar el entorno.


*Si deseas utilizar Jupyter Notebook u otro entorno de trabajo, puedes ignorar esta sección. En ese caso, te recomendamos consultar la documentación correspondiente para crear el entorno de trabajo adecuado:*

https://github.com/obspy/obspy/wiki#installation

Para instalar ObsPy, jedi y timezonefinder, se pueden utilizar los siguientes comandos a través del gestor de paquetes de Python (PyPI), de manera que no se ejecuten como líneas de codigo, si no como un comando de Shell.

Al terminar los comandos es necesario reinicar el entorno de ejecucion para que los cambios sean efectuados.

In [ ]:
! pip install obspy
! pip install jedi
! pip install timezonefinder

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 55.2 MB/s eta 0:00:00


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 40.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 91.9 MB/s eta 0:00:00
  Created wheel for timezonefinder: filename=timezonefinder-6.2.0-cp310-cp310-manylinux_2_31_x86_64.whl size=46903683 sha256=063be222c1183c0167409c575951b6c7c1add5d759f551a1fcbfd047ace78525
  Stored in directory: /root/.cache/pip/wheels/17/19/d4/ae94459b7f74f7e8f171862d1c08adedf9b7c76ddfc514a620
Successfully built timezonefinder


Se actulizan a la última versión disponible.

In [ ]:
! pip install -U obspy
! pip install -U timezonefinder

Montamos Google Drive para que Colab pueda importar y exportar documentos y archivos.


In [ ]:
# Darle permisos a Google Colab para que haga uso de Drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: ignored

Nos posicionamos en la carpeta de colab Notebooks

In [ ]:
cd drive/MyDrive/Colab Notebooks

/content/drive/MyDrive/Colab Notebooks


# Importar las bibliotecas necesarias

In [ ]:
import obspy
import glob
import os
import numpy as np
import pandas as pd
import timezonefinder
import seismosocialdistancing
import matplotlib.pylab as plt
import matplotlib.ticker as ticker
import tqdm
import warnings

from obspy.core.stream import Stream
from obspy import UTCDateTime, read, read_inventory
from obspy.signal import PPSD
from glob import glob
from itertools import zip_longest as zilo

# Parametros necesarios de las estaciones


Definimos inicio y final de los sets de datos.
Este caso se procesa un año a la vez, para llevar un control de los datos.

In [ ]:
start = UTCDateTime("2019, 01, 01")
end = UTCDateTime("2020, 01, 01")

# start = UTCDateTime("2020, 01, 01")
# end = UTCDateTime("2021, 01, 01")

# start = UTCDateTime("2021, 01, 01")
# end = UTCDateTime("2022, 01, 01")

# start = UTCDateTime("2022, 01, 01")
# end = UTCDateTime("2023, 01, 01")

datelist = pd.date_range(start.datetime, min(end, UTCDateTime()).datetime, freq="D")

Dentro de los parametros se pueden elegir dos rutas, una donde se encuentran los sets de datos y otra donde se exportaran los archivos csv, esta ruta puede ser la misma.

In [ ]:
#----- Seleccion de la estacion deseada -----#

#---- Estadio ----#
# station = "REBDD"
#---- Biblioteca ----#
station = "R6BB7"
#---- Casa de Cultura ----#
# station = "RF6B5"

network = "AM"
location = "00"
channel = "EHZ"
D = "D"
time_zone = 'America/Mexico_City'
year = str(start.year)    # Se extrae el año de start como identificador

data_path = "/content/drive/MyDrive/DataNew/{}/AM/{}/EHZ.D/".format(year, station)
csv_path = "/content/drive/MyDrive/data_csv/csv_%s/" % station


freqs = [(4.0,14.0),(4.0,20.0),(4.0,40.0),(6.0,18.0)]
scale = 1e9
nslc = "{}.{}.{}.{}.{}".format(network, station, location, channel, D)
tf = timezonefinder.TimezoneFinder()

plt.rcParams['figure.figsize'] = 15, 7
plt.rcParams['font.size']=12

# Carga de la respuesta instrumental y localización de las estaciones

Esto supone que se tiene el archivo .xml y se ubica en la caperta Colab Notebooks.

In [ ]:
inv = read_inventory('CGEO.xml')
inv = inv.select(station=station)
time_zone = tf.certain_timezone_at(lat=inv[0][0].latitude, lng=inv[0][0].longitude)
print(inv)

Inventory created at 2020-10-05T15:40:40.879242Z
	Sending institution: SeisComP3 (CGEO)
	Contains:
		Networks (1):
			AM
		Stations (1):
			AM.RF6B5 (Raspberry Shake Personal Seismograph Station)
		Channels (1):
			AM.RF6B5.00.EHZ


# Verificar la ruta de los datos y su disponibilidad

Tomando la ruta de los archivos directamente de Google Drive, observando la posible laguna de datos.

In [ ]:
1nslc = nslc.replace("*", "").replace("?", "")
pbar = tqdm.tqdm(datelist)
found_files = []
first_file_date = None

for day in pbar:
    datestr = day.strftime("%Y.%j")
    fn = "{}{}.{}".format(data_path, nslc, datestr)

    if day != UTCDateTime().datetime and os.path.isfile(fn):
        found_files.append(day)

total_files = len(datelist)
percentage = (len(found_files) / total_files) * 100

if found_files:
    first_file_date = found_files[0]
    last_file_date = found_files[-1]

print("\nFecha de inicio en UTC: ", first_file_date)
print("Fecha de termino en UTC: ", last_file_date)
print("Días encontrados: ", len(found_files))
print("Porcentaje de días útiles: {:.2f}%".format(percentage))

100%|██████████| 366/366 [00:03<00:00, 92.52it/s] 


Fecha de inicio en UTC:  2019-02-27 00:00:00
Fecha de termino en UTC:  2019-12-29 00:00:00
Días encontrados:  290
Porcentaje de días útiles: 79.23%


# Calculando Probabilistic Power Espectral Densities (PPSD)

Se usan parametros estandar, con una ventana de 30 min (1800s) y una superposicion del 50%, teniendo una resolucion de 15 minutos.

Para cada archivo MiniSeed se genera un .npz de salida, de esta forma se obtiene un PPSD diario y se evita saturar la memoria RAM.

In [ ]:
resp = inv

force_reprocess = False
pbar = tqdm.tqdm(datelist)
for day in pbar:
    datestr = day.strftime("%Y.%j")
    fn_in = "{}{}.{}".format(data_path,nslc, datestr)
    pbar.set_description("Processing %s" % fn_in)
    if not os.path.isfile(fn_in):
        continue
    stall = read(fn_in, headonly=True)
    for mseedid in list(set([tr.id for tr in stall])):
        fn_out = "{}{}_{}.npz".format(data_path, datestr, mseedid)
        if os.path.isfile(fn_out) and not force_reprocess:
            continue
        st = read(fn_in, sourcename=mseedid)
        st.attach_response(resp)
        ppsd = PPSD(st[0].stats, metadata=resp,
                    ppsd_length=1800, overlap=0.5,
                    period_smoothing_width_octaves=0.025,
                    period_step_octaves=0.0125,
                    period_limits=(0.008, 50),
                    db_bins=(-200, 20, 0.25))
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            ppsd.add(st)
        ppsd.save_npz(fn_out[:-4])
        del st, ppsd
    del stall

Processing /content/drive/MyDrive/DataNew/2019/AM/RF6B5/EHZ.D/AM.RF6B5.00.EHZ.D.2019.351:  96%|█████████▌| 350/366 [05:09<00:17,  1.10s/it]/usr/local/lib/python3.10/dist-packages/obspy/io/mseed/headers.py:823: InternalMSEEDWarning: readMSEEDBuffer(): Not a SEED record. Will skip bytes 12709376 to 12709503.
  warnings.warn(_w, InternalMSEEDWarning)
/usr/local/lib/python3.10/dist-packages/obspy/io/mseed/headers.py:823: InternalMSEEDWarning: readMSEEDBuffer(): Not a SEED record. Will skip bytes 12709504 to 12709631.
  warnings.warn(_w, InternalMSEEDWarning)
/usr/local/lib/python3.10/dist-packages/obspy/io/mseed/headers.py:823: InternalMSEEDWarning: readMSEEDBuffer(): Not a SEED record. Will skip bytes 12709632 to 12709759.
  warnings.warn(_w, InternalMSEEDWarning)
/usr/local/lib/python3.10/dist-packages/obspy/io/mseed/headers.py:823: InternalMSEEDWarning: readMSEEDBuffer(): Not a SEED record. Will skip bytes 12709760 to 12709887.
  warnings.warn(_w, InternalMSEEDWarning)
Processing /conte

# Volver a cargar los PPSD diarios

Se vuelven a cargar todos los archivos .npz en un solo objeto

In [ ]:
ppsds = {}
pbar = tqdm.tqdm(datelist)
for day in pbar:
    datestr = day.strftime("%Y.%j")
    fn_pattern = "{}{}_*.npz".format(data_path, datestr)
    pbar.set_description("Reading %s" % fn_pattern)
    for fn in glob(fn_pattern):
        mseedid = fn.replace(".npz", "").split("_")[-1]
        if mseedid not in ppsds:
            ppsds[mseedid] = PPSD.load_npz(fn)
        else:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ppsds[mseedid].add_npz(fn)

Reading /content/drive/MyDrive/DataNew/2019/AM/RF6B5/EHZ.D/2020.001_*.npz: 100%|██████████| 366/366 [00:18<00:00, 19.83it/s]


# Extrayendo RMS sísmico y espectrograma de potencia

Para extraer el RMS sismico se usa el scrip seismosociadistancing de Lecocq (@seismotom).

Este modulo obtiene la raiz cuadrada de la integracion discreta por la regla del trapecio de los espectros de potencia en los rangos de frecuencia que se establecieron en los parametros.

In [ ]:
displacement_RMS = {}

for mseedid, ppsd in tqdm.tqdm(ppsds.items()):
    ind_times = pd.DatetimeIndex([d.datetime for d in ppsd.current_times_used])
    df = pd.DataFrame(ppsd.psd_values, index=ind_times, columns=1./ppsd.period_bin_centers)
    df = df.sort_index(axis=1)
    displacement_RMS[mseedid] = seismosocialdistancing.df_rms(df, freqs, output="DISP")


# Concatenar todos los dataframes en uno solo
rms_utc = pd.concat(displacement_RMS.values())
filename_utc = "{}{}.{}.{}_UTC.csv".format(csv_path, station, channel, year)
rms_utc.to_csv(filename_utc)

El espectrograma se obtiene del objeto ppsds, extrayendo el rango completo de ruido.

In [ ]:
espectrograma = {}
freq_range = (1.0, 40.0)  # Rango de frecuencias deseado
data_espectro = {}

for mseedid, ppsd in tqdm.tqdm(ppsds.items()):
    ind_times = pd.DatetimeIndex([d.datetime for d in ppsd.current_times_used])
    df = pd.DataFrame(ppsd.psd_values, index=ind_times, columns = np.round(1. / ppsd.period_bin_centers * 2) / 2)
    df = df.sort_index(axis=1)
    espectrograma[mseedid] = df

for mseedid, df in tqdm.tqdm(espectrograma.items()):
    data_df = df.loc[:, (df.columns >= freq_range[0]) & (df.columns <= freq_range[1])]
    data_df = data_df.loc[:, ~data_df.columns.duplicated()]
    data_espectro[mseedid] = data_df

espectro_utc = pd.concat(data_espectro.values())
fil_espectro_utc = "{}{}.{}.{}_Espectro_UTC.csv".format(csv_path, station, channel, year)
espectro_utc.to_csv(fil_espectro_utc)

# Cambio a hora local

Se convierten los datos con hora UTC a la hora local y posteriormente se exportan en archivos csv, de estos archivos es donde se extraeran las figuras.

In [ ]:
rms_local = rms_utc.tz_localize('UTC').tz_convert(time_zone).tz_localize(None)
filename_local = "{}{}.{}.{}_Local.csv".format(csv_path, station, channel, year)
rms_local.to_csv(filename_local)

espectro_local = espectro_utc.tz_localize('UTC').tz_convert(time_zone).tz_localize(None)
fil_espectro_local = "{}{}.{}.{}_Espectro_Local.csv".format(csv_path, station, channel, year)
espectro_local.to_csv(fil_espectro_local)

# Control de calidad cualitativo

De todos los rangos de frecuencias se generan plots de previsualizacion de los datos, así como una media diurna y una media nocturna, para observar de serca el comportamiento general que se tiene.

In [ ]:
d = rms_local.copy()
for col in d.columns:
    plt.plot(d.index, d[col], label=col)
ticks = ticker.FuncFormatter(lambda x, pos: "{0:g}".format(x*scale))
plt.gca().yaxis.set_major_formatter(ticks)
plt.gcf().autofmt_xdate()
plt.gca().set_axisbelow(True)
plt.legend()
plt.show()

In [ ]:
tmp=d.copy()
tmp*=scale

prom_tmp = tmp.copy()
prom_tmp = prom_tmp.mean()
print(prom_tmp)

for i in tmp:
    prom_d = tmp[i].copy().between_time("6:00", "16:00")
    prom_d = prom_d.resample("1D").median().shift(12, "H")
    plt.plot(prom_d.index, prom_d, label=str(i) + ' diurno')
    plt.legend()

for i in tmp:
    prom_n = tmp[i].copy().between_time("23:00", "5:00")
    prom_n = prom_n.resample("1D").median().shift(12, "H")
    plt.plot(prom_n.index, prom_n, label=str(i) + ' nocturno')
    plt.legend()